# Flight Dataset — Descriptive Statistics
Saint-Exupery Airport | `database.db`

## 1. Import libraries

In [13]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Libraries ready')

Libraries ready


In [14]:
OUT_DIR = Path(os.getcwd()) / 'outputs' / 'descriptive_stats'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUT_DIR)


Output folder: c:\Users\ACER\Documents\GitHub\master-minds-challenge\outputs\descriptive_stats


## 2. Load data

In [15]:
_here = Path(os.getcwd())
DB_PATH = _here / 'database.db'
if not DB_PATH.exists():
    DB_PATH = _here /'database.db'

conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql_query('SELECT * FROM mouvements_aero_insa', conn)
conn.close()

df['LTScheduledDatetime'] = pd.to_datetime(df['LTScheduledDatetime'], errors='coerce')
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.dtypes

Loaded 364,623 rows x 195 columns


IdMovementVinci              str
IdFarms                   object
IdMovement                object
IdADL                        str
IdPkgStand                   str
                           ...  
PxScansRX                 object
PxScansGAT                object
PxScansSalonConfluence    object
PxScansSalonMontblanc     object
etl_origin                   str
Length: 195, dtype: object

## 3. Shape & column overview

In [16]:
summary = pd.DataFrame({
    'dtype'    : df.dtypes,
    'non_null' : df.notna().sum(),
    'null'     : df.isna().sum(),
    'null_%'   : (df.isna().mean() * 100).round(1),
    'unique'   : df.nunique(),
})
print(f'Shape: {df.shape}')
summary.to_csv(OUT_DIR / 'column_overview.csv')
print(f'Saved → {OUT_DIR / "column_overview.csv"}')
summary


Shape: (364623, 195)
Saved → c:\Users\ACER\Documents\GitHub\master-minds-challenge\outputs\descriptive_stats\column_overview.csv


,dtype,non_null,null,null_%,unique
IdMovementVinci,str,364623,0,0.00,310970
IdFarms,object,364623,0,0.00,310953
IdMovement,object,364623,0,0.00,348871
IdADL,str,364623,0,0.00,362390
IdPkgStand,str,364623,0,0.00,15
...,...,...,...,...,...
PxScansRX,object,364623,0,0.00,70
PxScansGAT,object,364623,0,0.00,314
PxScansSalonConfluence,object,364623,0,0.00,36
PxScansSalonMontblanc,object,364623,0,0.00,60


## 4. Numeric columns — statistics

In [17]:
# Convert likely-numeric columns stored as object
for col in df.columns:
    if df[col].dtype == object:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().mean() > 0.5:
            df[col] = converted

num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'Numeric columns ({len(num_cols)}): {num_cols}')

stats_df = df[num_cols].describe(percentiles=[.25, .5, .75, .90, .99]).T
stats_df['skewness'] = df[num_cols].skew().round(2)
stats_df['kurtosis'] = df[num_cols].kurt().round(2)
stats_df = stats_df.round(2)

stats_df.to_csv(OUT_DIR / 'numeric_stats.csv')
print(f'Saved → {OUT_DIR / "numeric_stats.csv"}')
stats_df


Numeric columns (41): ['IdFarms', 'IdTraficType', 'IdBusinessUnitType', 'IdBusContactType', 'IdTerminalType', 'IdDelayTypeDOPS', 'IdBagStatusDelivery', 'NbFlight', 'NbCounter', 'DelayHBLDurationMinutes', 'TaxiDurationMinutes', 'FarmsNbPax', 'FarmsNbPaxHTransit', 'FarmsNbPaxTotal', 'FarmsNbPaxPHMR', 'FarmsNbPaxAssisting', 'FarmsNbPaxExpected', 'NbAirbridge', 'InvoiceNbPayingPax', 'InvoiceNbPaxConnecting', 'InvoiceNbPaxTransit', 'InvoiceNbNonPayingPax', 'InvoiceNbPaxHTransit', 'InvoiceNbPaxTotal', 'InvoiceNbOfSeats', 'InvoiceNbOfNights', 'InvoiceMovementWeight', 'InvoiceFretWeight', 'InvoiceFretWeightTonnes', 'InvoiceMailWeight', 'NbOfSeats', 'NbPax', 'NbPaxHTransit', 'NbPaxTransit', 'NbPaxConnecting', 'NbPaxTotal', 'InvoicePkgDurationMinutes', 'InvoicePkgDurationMinutesDay', 'InvoicePkgDurationMinutesNight', 'IdDelayResponsableGroup', 'IdDelayAirportReason']
Saved → c:\Users\ACER\Documents\GitHub\master-minds-challenge\outputs\descriptive_stats\numeric_stats.csv


,count,mean,std,min,25%,50%,75%,90%,99%,max,skewness,kurtosis
IdFarms,311036.00,1016251279.90,1099404.01,1014351714.00,1015298543.75,1016205084.50,1017197618.25,1017830915.50,1018191930.65,1018213145.00,0.08,-1.15
IdTraficType,364623.00,1.12,0.42,0.00,1.00,1.00,1.00,2.00,2.00,2.00,0.74,1.78
IdBusinessUnitType,364623.00,1.25,1.09,1.00,1.00,1.00,1.00,1.00,9.00,10.00,5.62,34.18
IdBusContactType,364623.00,2.08,1.09,0.00,1.00,3.00,3.00,3.00,3.00,3.00,-0.47,-1.46
IdTerminalType,364623.00,0.92,0.27,0.00,1.00,1.00,1.00,1.00,1.00,1.00,-3.05,7.29
IdDelayTypeDOPS,311036.00,1.78,0.92,0.00,1.00,2.00,3.00,3.00,4.00,4.00,0.21,-1.05
IdBagStatusDelivery,364623.00,2.31,0.95,1.00,1.00,3.00,3.00,3.00,3.00,3.00,-0.64,-1.57
NbFlight,364623.00,1.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00
NbCounter,311036.00,2.72,3.57,0.00,0.00,0.00,6.00,9.00,10.00,17.00,0.90,-0.71
DelayHBLDurationMinutes,294874.00,13.32,40.18,-509.00,-5.00,3.00,19.00,46.00,160.00,5277.00,14.45,1136.78


## 5. Categorical columns — value counts

In [18]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}\n')

all_vc_rows = []
for col in cat_cols:
    vc  = df[col].value_counts(dropna=False).head(10)
    pct = (vc / len(df) * 100).round(1)
    tbl = pd.DataFrame({'count': vc, '%': pct})
    print(f'── {col} (unique={df[col].nunique()}) ──')
    print(tbl.to_string())
    print()

    # Collect for combined CSV
    for val, cnt, p in zip(vc.index, vc.values, pct.values):
        all_vc_rows.append({'column': col, 'value': val, 'count': cnt, '%': p})

cat_vc_df = pd.DataFrame(all_vc_rows)
cat_vc_df.to_csv(OUT_DIR / 'categorical_value_counts.csv', index=False)
print(f'Saved → {OUT_DIR / "categorical_value_counts.csv"}')


Categorical columns (153): ['IdMovementVinci', 'IdMovement', 'IdADL', 'IdPkgStand', 'IdIrregularityCode', 'IdRunway', 'IdAircraftType', 'IdDelayMainReasonSubcode', 'AirportCode', 'airlineOACICode', 'SysStopover', 'AirportOrigin', 'AirportPrevious', 'ServiceCode', 'flightNumber', 'OperatorFlightNumber', 'ExternalFlightNumber', 'FlightNumberNormalized', 'OperatorOACICodeNormalized', 'Counter', 'Conveyor', 'NbConveyor', 'LTEstimateDatetime', 'LTCancellationDatetime', 'LTActivationDatetime', 'LTBlockDatetime', 'LTScheduledTime', 'LTExternalDatetime', 'LTExternalDate', 'LTExternalTime', 'LTRunwayDatetime', 'LTFirstBagDeliveryDatetime', 'LTLastBagDeliveryDatetime', 'LTBagDeliveryDuration', 'LTFirstBagDeliveryDuration', 'LTLastBagDeliveryDuration', 'LTCtGInitialDatetime', 'LTCtGDynamicDatetime', 'LTCtGDisplayDatetime', 'LTCtGSecondCallDatetime', 'LTCtGLastCallDatetime', 'ResponsableModifCtG', 'UTCRunwayDatetime', 'UTCScheduledDatetime', 'UTCBlockDatetime', 'UTCExternalDate', 'UTCFirstBagDeliv

## 6. NbPaxTotal — detailed stats

In [19]:
df['NbPaxTotal'] = pd.to_numeric(df['NbPaxTotal'], errors='coerce')
pax = df['NbPaxTotal']

print('=== NbPaxTotal — full column ===')
print(f'  Count        : {pax.notna().sum():,}')
print(f'  Missing      : {pax.isna().sum():,}  ({pax.isna().mean()*100:.1f}%)')
print(f'  Zero         : {(pax == 0).sum():,}  ({(pax==0).mean()*100:.1f}%)')
print(f'  Negative     : {(pax < 0).sum():,}')

pax_valid = pax[pax > 0]
print('\n=== NbPaxTotal — valid flights (>0) ===')
print(f'  Count        : {len(pax_valid):,}')
print(f'  Mean         : {pax_valid.mean():.1f}')
print(f'  Median       : {pax_valid.median():.1f}')
print(f'  Std          : {pax_valid.std():.1f}')
print(f'  Min          : {pax_valid.min():.0f}')
print(f'  Max          : {pax_valid.max():.0f}')
print(f'  p25/p75/p99  : {pax_valid.quantile(0.25):.0f} / {pax_valid.quantile(0.75):.0f} / {pax_valid.quantile(0.99):.0f}')
print(f'  Skewness     : {pax_valid.skew():.3f}')
print(f'  Kurtosis     : {pax_valid.kurt():.3f}')

# Save NbPaxTotal summary
pax_summary = pd.DataFrame([{
    'count_non_null' : pax.notna().sum(),
    'missing'        : pax.isna().sum(),
    'missing_%'      : round(pax.isna().mean() * 100, 1),
    'zero'           : (pax == 0).sum(),
    'negative'       : (pax < 0).sum(),
    'valid_count'    : len(pax_valid),
    'mean'           : round(pax_valid.mean(), 1),
    'median'         : round(pax_valid.median(), 1),
    'std'            : round(pax_valid.std(), 1),
    'min'            : pax_valid.min(),
    'max'            : pax_valid.max(),
    'p25'            : pax_valid.quantile(0.25),
    'p75'            : pax_valid.quantile(0.75),
    'p99'            : pax_valid.quantile(0.99),
    'skewness'       : round(pax_valid.skew(), 3),
    'kurtosis'       : round(pax_valid.kurt(), 3),
}])
pax_summary.to_csv(OUT_DIR / 'nbpaxtotal_stats.csv', index=False)
print(f'\nSaved → {OUT_DIR / "nbpaxtotal_stats.csv"}')


=== NbPaxTotal — full column ===
  Count        : 311,036
  Missing      : 53,587  (14.7%)
  Zero         : 33,882  (9.3%)
  Negative     : 0

=== NbPaxTotal — valid flights (>0) ===
  Count        : 277,154
  Mean         : 120.1
  Median       : 128.0
  Std          : 54.7
  Min          : 1
  Max          : 1760
  p25/p75/p99  : 77 / 161 / 251
  Skewness     : 0.081
  Kurtosis     : 3.151

Saved → c:\Users\ACER\Documents\GitHub\master-minds-challenge\outputs\descriptive_stats\nbpaxtotal_stats.csv
